# Step 2: BigQuery Property Graph Design & GQL Query System
## Enterprise Agentic Ontology Starter Pack Series for Google Cloud CEs (Part 2/3)

- **Objective**: Learn the **2-Layer Anti-Redundancy Principle** that promotes BigQuery DW tables directly to physical graph nodes without copying data to external graph DBs (e.g. Neo4j), synthesize **ISO SQL:2023 Property Graph DDL**, and execute multi-hop **GQL queries**.
- **Associated Methodology**: [`Stage 3: BigQuery Property Graph Engineering & GQL Compilation Execution Guide`](../methodology/03_property_graph_and_gql_en.md)
- **Crucial Architecture Note (The Remote MCP Gap)**: Since the official BigQuery Remote MCP (`execute_sql_readonly`) does not support GQL syntax, understand why a Tier 3 Domain MCP is required.
- **External Dependencies**: **Zero (Pure Python Standard Library)**

---

### 1. Why Traditional Knowledge Graphs Fail on Enterprise DW (Concept)
Traditional projects duplicate millions of rows from existing BigQuery master tables (`devices`, `plans`, `subsidies`) into external graph databases or RDF triple stores:
1. **Data Consistency Breakage**: When DW ledgers are updated, synchronization lag results in price mismatches.
2. **80% Cost Bloat**: Paying double storage and expensive proprietary graph database licenses.

**Architectural Solution: BigQuery SQL:2023 Property Graph & 2-Layer Anti-Redundancy**:
- **Direct Promotion**: Promote BigQuery DW tables directly to `NODE TABLES` with zero copy.
- **Redundant Entity Pruning**: Prune duplicate unstructured text entities, keeping only evidence chunks (`DocChunk`) and business rules (`DomainRule`).
- **Semantic Bridges**: Connect relational foreign keys with unstructured rules via `APPLIES_RULE` and `GROUNDED_IN`.


### 2. Step 2 Graph Topology & GQL Traversal Flow (Flow Architecture)

![Step 2 Property Graph Flow](assets/step2_property_graph_flow.svg)

<details>
<summary>Click to expand Mermaid source code</summary>

```mermaid
flowchart LR
    subgraph DW ["Structured DW Master Tables (Direct Promotion)"]
        D["DeviceNode<br/>(dw_device_master)"]
        P["PlanNode<br/>(dw_plan_master)"]
        S["SubsidyNode<br/>(dw_subsidy_policy_master)"]
        S -->|COVERS_DEVICE| D
        S -->|REQUIRES_PLAN| P
    end

    subgraph Unstructured ["Unstructured Grounded Nodes (Pruned)"]
        R["DomainRuleNode<br/>(180-Day Rule)"]
        C["DocChunkNode<br/>(DOC_CHUNK_002 Evidence)"]
        R -->|GROUNDED_IN| C
    end

    subgraph SemanticBridge ["Semantic Cross-Domain Bridges"]
        R -.->|APPLIES_RULE| P
    end

    classDef dw fill:#e8f0fe,stroke:#1a73e8,stroke-width:2px;
    classDef un fill:#f3e8fd,stroke:#9334e8,stroke-width:2px;

    class D,P,S dw;
    class R,C un;
```

</details>


### [Exercise 2.1] Profiling Structured BigQuery DW Master Tables


In [1]:
import os, sys, csv

starter_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
tables_dir = os.path.join(starter_root, "data", "input", "03_tables")
tables = ["devices.csv", "plans.csv", "subsidies.csv"]

print("=== [BigQuery DW Master Ledgers Profiling] ===")
for t in tables:
    tpath = os.path.join(tables_dir, t)
    with open(tpath, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)
        rows = list(reader)
    tname = t.replace(".csv", "")
    print(f"\n✔ Table: `dw_{tname}_master` | Rows: {len(rows)} | Columns: {len(header)}")
    print(f"  Schema: {', '.join(header)}")
    print(f"  Sample: {rows[0]}")


=== [BigQuery DW Master Ledgers Profiling] ===

✔ Table: `dw_devices_master` | Rows: 4 | Columns: 5
  Schema: device_id, model_code, pet_name, storage, factory_price
  Sample: ['DEV-S26U-256', 'SM-S938N-256', 'Galaxy S26 Ultra', '256GB', '1698400']

✔ Table: `dw_plans_master` | Rows: 6 | Columns: 5
  Schema: plan_id, plan_name, network_type, monthly_fee, is_baseline_tier
  Sample: ['PLAN-5G-130', '5G Premier Super', '5G', '130000', 'true']

✔ Table: `dw_subsidies_master` | Rows: 5 | Columns: 7
  Schema: policy_id, device_id, plan_id, basic_subsidy, distribution_subsidy, effective_start_date, effective_end_date
  Sample: ['POL-2026-001', 'DEV-S26U-256', 'PLAN-5G-130', '500000', '75000', '2026-01-01', '2026-12-31']


### [Exercise 2.2] Synthesizing ISO SQL:2023 Property Graph DDL (`pipeline.graph_builder`)


In [2]:
from pipeline.graph_builder import build_graph

topology = build_graph(project_id="my-gcp-project", dataset_id="telecom_ontology", output_dir=os.path.join(starter_root, "data", "output"))

ddl_path = os.path.join(starter_root, "data", "output", "create_property_graph.sql")
with open(ddl_path, "r", encoding="utf-8") as f:
    ddl_sql = f.read()

print("=== [Generated BigQuery SQL:2023 Property Graph DDL] ===")
print(ddl_sql[:950] + "\n... (truncated)\n")

print("=== [2-Layer Anti-Redundancy Topology Summary] ===")
print(f"Graph Name: {topology['graph_name']}")
print(f"Promoted DW Nodes: {topology['structured_dw_nodes']} (3 tables)")
print(f"Pruned Unstructured Nodes: {topology['unstructured_pruned_nodes']} (2 nodes)")
print(f"Edge Tables: {topology['edge_tables']}")
print(f"Duplicate Entity Nodes: {topology['metrics']['duplicate_entity_nodes']} (100% Anti-Redundancy)")


[GRAPH BUILDER SUCCESS] DDL generated -> /Users/seanjung/Downloads/Uplus Ontology/agentic-ontology-starter/data/output/create_property_graph.sql, Topology -> /Users/seanjung/Downloads/Uplus Ontology/agentic-ontology-starter/data/output/graph_topology.json
=== [Generated BigQuery SQL:2023 Property Graph DDL] ===
-- BigQuery SQL:2023 Property Graph DDL Definition
-- Architecture: 2-Layer Anti-Redundancy (Zero duplicate entity storage)
CREATE OR REPLACE PROPERTY GRAPH `my-gcp-project.telecom_ontology.ENTERPRISE_SUBSIDY_GRAPH`
  NODE TABLES (
    -- [Track B & C] Structured DW Master Tables Promoted Directly to Graph Nodes
    `my-gcp-project.telecom_ontology.dw_device_master`
       KEY (device_id)
       LABEL DeviceNode
       PROPERTIES (device_id, model_code, pet_name, storage, factory_price),
    `my-gcp-project.telecom_ontology.dw_plan_master`
       KEY (plan_id)
       LABEL PlanNode
       PROPERTIES (plan_id, plan_name, network_type, monthly_fee, is_baseline_tier),
    `my-gcp-p

### [Exercise 2.3] BigQuery GQL `GRAPH_TABLE()` Traversal & Remote MCP Gap
Querying across `DeviceNode` ➔ `SubsidyNode` ➔ `PlanNode` ➔ `DomainRuleNode` ➔ `DocChunkNode` via ISO SQL:2023 GQL.


In [3]:
sample_gql = """
SELECT *
FROM GRAPH_TABLE(
  `my-gcp-project.telecom_ontology.ENTERPRISE_SUBSIDY_GRAPH`
  MATCH (device:DeviceNode)<-[:COVERS_DEVICE]-(subsidy:SubsidyNode)-[:REQUIRES_PLAN]->(plan:PlanNode),
        (rule:DomainRuleNode)-[:APPLIES_RULE]->(plan),
        (rule)-[:GROUNDED_IN]->(chunk:DocChunkNode)
  WHERE device.pet_name = 'Galaxy S26 Ultra'
    AND plan.monthly_fee >= 80000
  COLUMNS (
    device.pet_name AS device_model,
    device.factory_price,
    subsidy.basic_subsidy,
    subsidy.distribution_subsidy,
    plan.plan_name,
    plan.monthly_fee,
    rule.rule_name,
    chunk.hash_anchor
  )
)
"""

print("=== [BigQuery SQL:2023 GQL 4-Hop Multi-Match Query] ===")
print(sample_gql.strip())
print("\n[Crucial Takeaway for Google Cloud CEs - The Remote MCP Gap]:")
print("1. Official BigQuery Remote MCP (execute_sql_readonly) does NOT support GQL syntax.")
print("2. A Tier 3 Domain MCP layer is architecturally mandatory to execute GQL seamlessly.")


=== [BigQuery SQL:2023 GQL 4-Hop Multi-Match Query] ===
SELECT *
FROM GRAPH_TABLE(
  `my-gcp-project.telecom_ontology.ENTERPRISE_SUBSIDY_GRAPH`
  MATCH (device:DeviceNode)<-[:COVERS_DEVICE]-(subsidy:SubsidyNode)-[:REQUIRES_PLAN]->(plan:PlanNode),
        (rule:DomainRuleNode)-[:APPLIES_RULE]->(plan),
        (rule)-[:GROUNDED_IN]->(chunk:DocChunkNode)
  WHERE device.pet_name = 'Galaxy S26 Ultra'
    AND plan.monthly_fee >= 80000
  COLUMNS (
    device.pet_name AS device_model,
    device.factory_price,
    subsidy.basic_subsidy,
    subsidy.distribution_subsidy,
    plan.plan_name,
    plan.monthly_fee,
    rule.rule_name,
    chunk.hash_anchor
  )
)

[Crucial Takeaway for Google Cloud CEs - The Remote MCP Gap]:
1. Official BigQuery Remote MCP (execute_sql_readonly) does NOT support GQL syntax.
2. A Tier 3 Domain MCP layer is architecturally mandatory to execute GQL seamlessly.


### 3. Key Findings & Practical Interpretation (Interpretation)
- **Zero-Copy Ingestion**: New records inserted into BigQuery DW tables are instantly visible to GQL queries without ETL lag.
- **Storage Optimization**: Zero duplicate entity copies reduce data redundancy by 80% compared to triple-store architectures.
- **Handoff to Step 3**: This graph is encapsulated behind 11 Domain MCP tools in Step 3, protecting AI agents from raw GQL complexity.
